# IMPORTS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import time
import os
import copy
from torch.utils import data
from PIL import Image
import random
import cv2
import json
import datetime
import multiprocessing
import queue
import threading
import lightning.pytorch as pl

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_Weights # ADDED THIS LINE

from torchvision.datasets import CocoDetection
from torch.utils.data import DataLoader, Dataset

from pycocotools.coco import COCO

import os

to_tensor = torchvision.transforms.ToTensor()

print("PyTorch Version: ",torch.__version__)
print("Torchvision Version: ",torchvision.__version__)

PyTorch Version:  2.11.0+cpu
Torchvision Version:  0.26.0+cpu


In [2]:
# Custom PyTorch Dataset to load COCO-format annotations and images
class CocoDetectionDataset(Dataset):
    # Init function: loads annotation file and prepares list of image IDs
    def __init__(self, image_dir, annotation_path, transforms=None):
        self.image_dir = image_dir
        self.coco = COCO(annotation_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transforms = transforms

    # Returns total number of images
    def __len__(self):
        return len(self.image_ids)

    # Fetches a single image and its annotations
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.coco.loadImgs(image_id)[0]
        image_path = os.path.join(self.image_dir, image_info['file_name'])
        image = Image.open(image_path).convert("RGB")

        # Load all annotations for this image
        annotation_ids = self.coco.getAnnIds(imgIds=image_id)
        annotations = self.coco.loadAnns(annotation_ids)

        # Extract bounding boxes and labels from annotations
        boxes = []
        labels = []
        for obj in annotations:
            xmin, ymin, width, height = obj['bbox']
            xmax = xmin + width
            ymax = ymin + height
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(obj['category_id'])

        # Convert annotations to PyTorch tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        area = torch.as_tensor([obj['area'] for obj in annotations], dtype=torch.float32)
        iscrowd = torch.as_tensor([obj.get('iscrowd', 0) for obj in annotations], dtype=torch.int64)

        # Package everything into a target dictionary
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        # Apply transforms if any were passed
        if self.transforms:
            image = self.transforms(image)

        return image, target

In [3]:
# Data module for pytorch lightning
class RCNNDataModule(pl.LightningDataModule):
    def __init__(self,
                 train_image_dir,
                 train_annotation_path,
                 val_image_dir,
                 val_annotation_path,
                 batch_size=4,
                 num_workers=4,
                 **kwargs):
        super().__init__(**kwargs)
        self.train_image_dir = train_image_dir
        self.train_annotation_path = train_annotation_path
        self.val_image_dir = val_image_dir
        self.val_annotation_path = val_annotation_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.data_train = None
        self.data_val = None

    def setup(self, stage: str):
        if (stage == 'fit' or stage == 'validate' and not(self.data_train and self.data_val)):
            # Load up the images
            self.data_train = CocoDetectionDataset(image_dir=self.train_image_dir,
                                                   annotation_path=self.train_annotation_path,
                                                   transforms=transforms.ToTensor())

            self.data_val = CocoDetectionDataset(image_dir=self.val_image_dir,
                                                 annotation_path=self.val_annotation_path,
                                                 transforms=transforms.ToTensor())

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.data_train,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=True,
                                           collate_fn=lambda x: tuple(zip(*x)))

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.data_val,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=False,
                                           collate_fn=lambda x: tuple(zip(*x)))

In [4]:
# The actual network. It's essentially a wrapper for the faster rcnn network
class RCNN(pl.LightningModule):
    def __init__(self,
                 num_classes: int,
                 **kwargs):
        super().__init__()

        # Use DEFAULT weights explicitly
        self.rcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights)

        in_features = self.rcnn.roi_heads.box_predictor.cls_score.in_features
        self.rcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    def forward(
        self,
        input_images: list[torch.Tensor],
        input_data: list[dict[str, torch.Tensor]] | None = None
    ):
        return self.rcnn(input_images, input_data)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=0.001)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 5) # Make sure to change the number if you're doing more/less than 5 epochs

        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler,
            "gradient_clip_val": 1.0
        }

    def training_step(self, train_batch, batch_idx):
        imgs, annotations_batch = train_batch
        input_images = [img.to(self.device) for img in imgs]
        input_data = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in annotations_batch]

        self.train()
        loss_dict = self(input_images, input_data)
        loss = sum(loss for loss in loss_dict.values())

        self.log('train_loss', loss, on_step=True, on_epoch=True)

        return loss

    def validation_step(self, val_batch, batch_idx):
        imgs, annotations_batch = val_batch
        input_images = [img.to(self.device) for img in imgs]
        input_data = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in annotations_batch]

        self.train()
        with torch.no_grad():
            loss_dict = self(input_images, input_data)
        loss = sum(loss for loss in loss_dict.values())

        self.log('val_loss', loss, on_step=True, on_epoch=True)

        return loss

In [ ]:

data_module = RCNNDataModule(train_image_dir="dataset/images/train",
                             train_annotation_path="dataset/instances_train.json",
                             val_image_dir="dataset/images/val",
                             val_annotation_path="dataset/instances_val.json")

In [ ]:
# Make sure the # of classes match the data. In the future, this should be done automatically from the .json coco file
model = RCNN(num_classes=57, device="cuda")

c:\Users\Philip\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# Define the trainer
trainer = pl.Trainer(logger=None,
                     max_epochs=5,
                     enable_progress_bar=True,
                     log_every_n_steps=0,
                     callbacks=[pl.callbacks.RichProgressBar(refresh_rate=20)])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
# This line does all of the training
trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('AMD Radeon RX 6800') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


┏━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ rcnn │ FasterRCNN │ 41.4 M │ train │     0 │
└───┴──────┴────────────┴────────┴───────┴───────┘

Trainable params: 41.2 M                                                                                           
Non-trainable params: 222 K                                                                                        
Total params: 41.4 M                                                                                               
Total estimated model params size (MB): 165                                                                        
Modules in train mode: 189                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Philip\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\lightning\pytorch\trainer\connectors\data_conne
ctor.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker 
initialization.

In [ ]:
# Save the weights to be loaded later
# Note: these weights (the .pth file) can't be used with C++. This is essentially for reloading the model in python
torch.save(model.state_dict(), "faster_rcnn_weights.pth")

# Loading model from save

In [ ]:
# Reload the model from scratch
model = RCNN(num_classes=29)

c:\Users\Philip\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
# Load the weights from the file
model.load_state_dict(torch.load("faster_rcnn_weights.pth"))

<All keys matched successfully>

In [10]:
from torchview import draw_graph

model.eval()
model_graph = draw_graph(model, 
                         input_size=(1, 3, 384, 384), 
                         depth=20, # Lower to see less detail in the graph and increase it to see more.
                         expand_nested=True, 
                         hide_module_functions=False)

In [11]:
# Warning: the graph is really big. Like, REALLY big.
model_graph.visual_graph.render(filename="faster-rcnn-visualization", format="svg", cleanup=True)


(process:82500): Pango-WARNING **: 21:03:31.130: couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.


'faster-rcnn-visualization.svg'

In [ ]:
# Export model to ONNX format (for use in C++ with ONNX runtime)
dummy_input = torch.randn(1, 3, 384, 384).to("cpu")
model = model.to("cpu")

torch.onnx.export(
    model,
    dummy_input,
    "faster_rcnn.onnx",
    opset_version=11, 
    input_names=["images"],
    output_names=["boxes", "labels", "scores"],
    dynamo=False,
    dynamic_axes=None
)

C:\Users\Philip\AppData\Local\Temp\ipykernel_35360\10071386.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
c:\Users\Philip\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\torch\nn\functional.py:4701: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=